# (노트) 판다스 - tidydata

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [파이썬]

### imports

In [804]:
import pandas as pd 
import numpy as np
from plotnine import * 
import matplotlib.pyplot as plt

### tidy data 

`-` 느낌: ggplot으로 그림그리기 좋은 데이터 + pandas로 query, groupy by 등을 쓰기 좋은 자료

`-` 정의: (1) col: variable (2) row: observation (3) Each value must have its own cell.
- https://r4ds.had.co.nz/tidy-data.html

예시1

|obs|x|y|shape|color|
|:-:|:-:|:-:|:-:|:-:|
|0|0|0 |'star'|'F'|
|1|0|1 |'circ'|'F'|
|2|1|0 |'star'|'M'|
|3|1|1 |'circ'|'M'|

예시2

| |shape=star|shape=circ|
|:-:|:-:|:-:|
|color=F|(0,0)|(0,1)|
|color=M|(1,0)|(1,1)|



### 예제1: state_fruit 

https://github.com/PacktPublishing/Pandas-Cookbook-Second-Edition/blob/master/data/state_fruit.csv

`-` 문제의 깃헙주소로 들어가서 데이터를 관찰 $\to$ 좌측상단이 'State'로 채워져있음 $\to$ 데이터를 읽을때 `index_col=0` 옵션을 사용함. 

In [764]:
url = 'https://raw.githubusercontent.com/PacktPublishing/Pandas-Cookbook-Second-Edition/master/data/state_fruit.csv'
df = pd.read_csv(url,index_col=0) 
df

,Apple,Orange,Banana
Texas,12,10,40
Arizona,9,7,12
Florida,0,14,190


#### 풀이1: stack + reset_index

In [765]:
df.stack()

Texas    Apple      12
         Orange     10
         Banana     40
Arizona  Apple       9
         Orange      7
         Banana     12
Florida  Apple       0
         Orange     14
         Banana    190
dtype: int64

In [766]:
df.stack().reset_index()

,level_0,level_1,0
0,Texas,Apple,12
1,Texas,Orange,10
2,Texas,Banana,40
3,Arizona,Apple,9
4,Arizona,Orange,7
5,Arizona,Banana,12
6,Florida,Apple,0
7,Florida,Orange,14
8,Florida,Banana,190


In [767]:
df.stack().reset_index().rename(columns={'level_0':'G1','level_1':'G2',0:'X1'})

,G1,G2,X1
0,Texas,Apple,12
1,Texas,Orange,10
2,Texas,Banana,40
3,Arizona,Apple,9
4,Arizona,Orange,7
5,Arizona,Banana,12
6,Florida,Apple,0
7,Florida,Orange,14
8,Florida,Banana,190


`-` 이름을 바꾸는 또다른 방법

In [768]:
_tidy= df.stack().reset_index()
_tidy

,level_0,level_1,0
0,Texas,Apple,12
1,Texas,Orange,10
2,Texas,Banana,40
3,Arizona,Apple,9
4,Arizona,Orange,7
5,Arizona,Banana,12
6,Florida,Apple,0
7,Florida,Orange,14
8,Florida,Banana,190


In [769]:
_tidy.columns

Index(['level_0', 'level_1', 0], dtype='object')

In [770]:
pd.Index(['G1','G2','X1'])

Index(['G1', 'G2', 'X1'], dtype='object')

In [771]:
_tidy.columns=pd.Index(['G1','G2','X1'])

In [772]:
_tidy

,G1,G2,X1
0,Texas,Apple,12
1,Texas,Orange,10
2,Texas,Banana,40
3,Arizona,Apple,9
4,Arizona,Orange,7
5,Arizona,Banana,12
6,Florida,Apple,0
7,Florida,Orange,14
8,Florida,Banana,190


`-` 이름을 바꾸는 잘못된 방법: 열이름을 변경할때 df.columns.values를 이용하여 억지로 바꾸면 문제가 생긴다. 

In [773]:
_tidy= df.stack().reset_index()
_tidy

,level_0,level_1,0
0,Texas,Apple,12
1,Texas,Orange,10
2,Texas,Banana,40
3,Arizona,Apple,9
4,Arizona,Orange,7
5,Arizona,Banana,12
6,Florida,Apple,0
7,Florida,Orange,14
8,Florida,Banana,190


In [774]:
_tidy.columns

Index(['level_0', 'level_1', 0], dtype='object')

In [775]:
_tidy.columns.values

array(['level_0', 'level_1', 0], dtype=object)

In [776]:
_tidy.columns.values[0]

'level_0'

In [777]:
_tidy.columns.values[0]='G1'
_tidy.columns.values[1]='G2'
_tidy.columns.values[2]='X3'

In [778]:
_tidy.columns

Index(['G1', 'G2', 'X3'], dtype='object')

- 얼핏보면 잘 바뀐듯 보이는데 key로 인덱싱이 안된다. (한마디로 위의 데이터프레임은 깨져있다.) 

`-` KeyError를 확인하자. 

In [779]:
_tidy['G1']

KeyError: 'G1'

#### 풀이2: stack + rename_axis + reset_index

In [780]:
df.stack()

Texas    Apple      12
         Orange     10
         Banana     40
Arizona  Apple       9
         Orange      7
         Banana     12
Florida  Apple       0
         Orange     14
         Banana    190
dtype: int64

In [781]:
df.stack().rename_axis(['G1','G2']) 

G1       G2    
Texas    Apple      12
         Orange     10
         Banana     40
Arizona  Apple       9
         Orange      7
         Banana     12
Florida  Apple       0
         Orange     14
         Banana    190
dtype: int64

In [782]:
df.stack().rename_axis(['G1','G2']).reset_index()

,G1,G2,0
0,Texas,Apple,12
1,Texas,Orange,10
2,Texas,Banana,40
3,Arizona,Apple,9
4,Arizona,Orange,7
5,Arizona,Banana,12
6,Florida,Apple,0
7,Florida,Orange,14
8,Florida,Banana,190


In [783]:
df.stack().rename_axis(['G1','G2']).reset_index().rename(columns={0:'X1'})

,G1,G2,X1
0,Texas,Apple,12
1,Texas,Orange,10
2,Texas,Banana,40
3,Arizona,Apple,9
4,Arizona,Orange,7
5,Arizona,Banana,12
6,Florida,Apple,0
7,Florida,Orange,14
8,Florida,Banana,190


#### 풀이3: melt(id_var=[??])

`-` `index=0`을 사용하지 않고 데이터를 불러왔을 경우 

In [784]:
df2 = pd.read_csv(url)  #state_fruit = pd.read_csv(url,index_col=0) 
df2

,Unnamed: 0,Apple,Orange,Banana
0,Texas,12,10,40
1,Arizona,9,7,12
2,Florida,0,14,190


In [785]:
df2.rename(columns={'Unnamed: 0':'G1'})

,G1,Apple,Orange,Banana
0,Texas,12,10,40
1,Arizona,9,7,12
2,Florida,0,14,190


In [786]:
df2.rename(columns={'Unnamed: 0':'G1'}).melt(id_vars=['G1'])

,G1,variable,value
0,Texas,Apple,12
1,Arizona,Apple,9
2,Florida,Apple,0
3,Texas,Orange,10
4,Arizona,Orange,7
5,Florida,Orange,14
6,Texas,Banana,40
7,Arizona,Banana,12
8,Florida,Banana,190


In [787]:
df2.rename(columns={'Unnamed: 0':'G1'}).melt(id_vars=['G1']).rename(columns={'variable':'G2','value':'X1'})

,G1,G2,X1
0,Texas,Apple,12
1,Arizona,Apple,9
2,Florida,Apple,0
3,Texas,Orange,10
4,Arizona,Orange,7
5,Florida,Orange,14
6,Texas,Banana,40
7,Arizona,Banana,12
8,Florida,Banana,190


***melt를 이용하여 데이터를 깨끗하게 만들기 위한 작전 (변형하고 싶은 데이터의 형태가 직관적으로 떠오르지 않을 경우)***

`*` Step1: [Unnamed: 0, Apple, Orange, Banana] 중 하나의 변수를 선택하여 index로 설정한다. 
- 요령: [Unnamed: 0, Apple, Orange, Banana] 중에서 숫자를 값으로 가지는 변수명은 제외한다. $\to$ 여기에서 Apple, Orange, Banana가 제외됨. 

`*` Step2: melt()사용. 필수옵션은 `id_vars=['Unnamed: 0']`
- `id_vars`: Step1에서 선택한 변수이름, 즉 State. `G1`에 해당하는 변수명. 

`*` Step3: 나머지 변수를 적당히 이름붙인다. 

#### 풀이4: 틀린풀이

In [788]:
df2

,Unnamed: 0,Apple,Orange,Banana
0,Texas,12,10,40
1,Arizona,9,7,12
2,Florida,0,14,190


In [789]:
df2.stack()

0  Unnamed: 0      Texas
   Apple              12
   Orange             10
   Banana             40
1  Unnamed: 0    Arizona
   Apple               9
   Orange              7
   Banana             12
2  Unnamed: 0    Florida
   Apple               0
   Orange             14
   Banana            190
dtype: object

`-` 여기서 어떻게 하죠? 

`-` 다루기 까다로워짐. 즉 망한것임 

#### 풀이5: set_index + stack + reset_index

`-` df에서는 stack을 쓸 수 있지만 df2에서는 stack을 쓰기 까다롭다. 

`-` df, df2의 차이점이 무엇인가?

In [790]:
display(df)
display(df2)

,Apple,Orange,Banana
Texas,12,10,40
Arizona,9,7,12
Florida,0,14,190


,Unnamed: 0,Apple,Orange,Banana
0,Texas,12,10,40
1,Arizona,9,7,12
2,Florida,0,14,190


In [791]:
df.index, df2.index

(Index(['Texas', 'Arizona', 'Florida'], dtype='object'),
 RangeIndex(start=0, stop=3, step=1))

`-` 즉 인덱스의 차이

`-` df2에서도 df1과 같이 인덱스를 만들어주면 stack을 쓸 수 있다.

In [792]:
df2.set_index('Unnamed: 0')

,Apple,Orange,Banana
Unnamed: 0,,,
Texas,12,10,40
Arizona,9,7,12
Florida,0,14,190


In [793]:
df2.set_index('Unnamed: 0').stack()

Unnamed: 0        
Texas       Apple      12
            Orange     10
            Banana     40
Arizona     Apple       9
            Orange      7
            Banana     12
Florida     Apple       0
            Orange     14
            Banana    190
dtype: int64

In [794]:
df2.set_index('Unnamed: 0').stack().reset_index()

,Unnamed: 0,level_1,0
0,Texas,Apple,12
1,Texas,Orange,10
2,Texas,Banana,40
3,Arizona,Apple,9
4,Arizona,Orange,7
5,Arizona,Banana,12
6,Florida,Apple,0
7,Florida,Orange,14
8,Florida,Banana,190


In [795]:
df2.set_index('Unnamed: 0').stack().reset_index().rename(columns={'Unnamed: 0':'G1','level_1':'G2',0:'X1'})

,G1,G2,X1
0,Texas,Apple,12
1,Texas,Orange,10
2,Texas,Banana,40
3,Arizona,Apple,9
4,Arizona,Orange,7
5,Arizona,Banana,12
6,Florida,Apple,0
7,Florida,Orange,14
8,Florida,Banana,190


#### 풀이6: reset_index + melt

In [802]:
df

,Apple,Orange,Banana
Texas,12,10,40
Arizona,9,7,12
Florida,0,14,190


In [803]:
df.reset_index().melt('index').rename(columns={'index':'G1','variable':'G2','value':'X1'})

,G1,G2,X1
0,Texas,Apple,12
1,Arizona,Apple,9
2,Florida,Apple,0
3,Texas,Orange,10
4,Arizona,Orange,7
5,Florida,Orange,14
6,Texas,Banana,40
7,Arizona,Banana,12
8,Florida,Banana,190
